In [1]:
!pip install torch-pruning scipy tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 5.1 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from torchvision.models import resnet50
from tqdm import tqdm
import numpy as np
from scipy.stats import spearmanr
import copy
import torch_pruning as tp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 128
EPOCHS_PRETRAIN = 100
EPOCHS_FINETUNE = 100
PRUNE_RATIO = 0.3

CALIB_SAMPLES = 2000
ORACLE_BATCHES = 5

LR = 0.1
WEIGHT_DECAY = 5e-4

print("Device:", DEVICE)

Device: cuda


In [3]:
train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914,0.4822,0.4465),
                (0.2470,0.2435,0.2616))
])

test_tf = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914,0.4822,0.4465),
                (0.2470,0.2435,0.2616))
])

train_set = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=train_tf
)

test_set = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=test_tf
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

100%|██████████| 169M/169M [00:01<00:00, 84.7MB/s]


In [4]:
model = resnet50(num_classes=100)
model.conv1 = nn.Conv2d(3,64,3,1,1,bias=False)
model.maxpool = nn.Identity()
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                            momentum=0.9, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[25,35], gamma=0.1)

def evaluate(model):
    model.eval()
    correct=0
    total=0
    with torch.no_grad():
        for x,y in test_loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x).argmax(1)
            correct += (pred==y).sum().item()
            total += y.size(0)
    return correct/total
best_accu=0
print("Starting Pretraining...")
for epoch in range(EPOCHS_PRETRAIN):
    model.train()
    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x),y)
        loss.backward()
        optimizer.step()
    scheduler.step()
    acc = evaluate(model)
    print(f"Epoch {epoch+1} | Acc {acc*100:.2f}%")
    if acc > best_accu :
        best_accu = acc
        print("BEST ACCURACY ----------------------------  ")

Starting Pretraining...
Epoch 1 | Acc 2.46%
BEST ACCURACY ----------------------------  
Epoch 2 | Acc 4.54%
BEST ACCURACY ----------------------------  
Epoch 3 | Acc 8.00%
BEST ACCURACY ----------------------------  
Epoch 4 | Acc 13.61%
BEST ACCURACY ----------------------------  
Epoch 5 | Acc 16.28%
BEST ACCURACY ----------------------------  
Epoch 6 | Acc 21.35%
BEST ACCURACY ----------------------------  
Epoch 7 | Acc 24.37%
BEST ACCURACY ----------------------------  
Epoch 8 | Acc 28.17%
BEST ACCURACY ----------------------------  
Epoch 9 | Acc 32.22%
BEST ACCURACY ----------------------------  
Epoch 10 | Acc 33.86%
BEST ACCURACY ----------------------------  
Epoch 11 | Acc 38.15%
BEST ACCURACY ----------------------------  
Epoch 12 | Acc 37.68%
Epoch 13 | Acc 41.20%
BEST ACCURACY ----------------------------  
Epoch 14 | Acc 43.56%
BEST ACCURACY ----------------------------  
Epoch 15 | Acc 37.09%
Epoch 16 | Acc 38.76%
Epoch 17 | Acc 46.31%
BEST ACCURACY ---------------

In [5]:
idx = torch.randperm(len(train_set))[:CALIB_SAMPLES]
calib_set = Subset(train_set, idx)
calib_loader = DataLoader(calib_set, batch_size=64)

activations = {}
gradients = {}

def fwd_hook(name):
    def hook(m,i,o):
        activations[name] = o.detach()
    return hook

def bwd_hook(name):
    def hook(m,gi,go):
        gradients[name] = go[0].detach()
    return hook

for name,m in model.named_modules():
    if isinstance(m, nn.Conv2d):
        m.register_forward_hook(fwd_hook(name))
        m.register_full_backward_hook(bwd_hook(name))
        
taylor = {}
iafb = {}

model.eval()
print("Calculating Filter Importance...")
for x,y in tqdm(calib_loader):
    x,y = x.to(DEVICE), y.to(DEVICE)
    out = model(x)
    loss = criterion(out,y)
    model.zero_grad()
    loss.backward()

    for k in activations:
        A = activations[k]
        G = gradients[k]
        score = torch.abs(A*G).sum((2,3))
        taylor.setdefault(k,[]).append(score.cpu())
        iafb.setdefault(k,[]).append(score.cpu())
        
taylor_final = {}
iafb_final = {}

for k in taylor:
    v = torch.cat(taylor[k],0)
    taylor_final[k] = v.mean(0)
    iafb_final[k] = v.median(0).values

Calculating Filter Importance...


100%|██████████| 32/32 [00:07<00:00,  4.27it/s]


In [6]:
def compute_oracle(model, layer_name):
    layer = dict(model.named_modules())[layer_name]
    num_filters = layer.weight.shape[0]

    # baseline loss
    base_loss = 0
    count=0
    
    with torch.no_grad(): # Added no_grad to save memory and compute
        for x,y in list(calib_loader)[:ORACLE_BATCHES]:
            x,y = x.to(DEVICE), y.to(DEVICE)
            base_loss += criterion(model(x),y).item()
            count+=1
        base_loss /= count

        oracle_scores = []

        for f in range(num_filters):
            original = layer.weight.data[f].clone()
            layer.weight.data[f] = 0

            loss=0
            count=0
            for x,y in list(calib_loader)[:ORACLE_BATCHES]:
                x,y = x.to(DEVICE), y.to(DEVICE)
                loss += criterion(model(x),y).item()
                count+=1
            loss/=count

            oracle_scores.append(abs(loss-base_loss))
            layer.weight.data[f]=original

    return torch.tensor(oracle_scores)

print("\n===== SPEARMAN CORRELATION =====")

# Specify exactly which layers to check here
target_layers = ["conv1", "layer1.0.conv1", "layer1.0.conv2"]

for k in target_layers:
    if k not in taylor_final:
        print(f"Skipping {k} - not found in taylor_final")
        continue

    oracle = compute_oracle(model, k)

    t_corr = spearmanr(
        oracle.numpy(),
        taylor_final[k].numpy()
    ).correlation

    i_corr = spearmanr(
        oracle.numpy(),
        iafb_final[k].numpy()
    ).correlation

    print(f"{k} | Taylor: {t_corr:.3f} | IAFB: {i_corr:.3f}")

def count_params(m):
    return sum(p.numel() for p in m.parameters())

params_before = count_params(model)
print("\nParameters before pruning:", params_before)
BASELINE_SAVE_PATH = "/kaggle/working/resnet50_cifar100_baseline100epochs.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "params": count_params(model)
}, BASELINE_SAVE_PATH)

print("✅ Baseline model saved at:", BASELINE_SAVE_PATH)


===== SPEARMAN CORRELATION =====
conv1 | Taylor: 0.537 | IAFB: 0.522
layer1.0.conv1 | Taylor: 0.240 | IAFB: 0.243
layer1.0.conv2 | Taylor: 0.383 | IAFB: 0.384

Parameters before pruning: 23705252
✅ Baseline model saved at: /kaggle/working/resnet50_cifar100_baseline100epochs.pt


In [7]:
def prune_structural(model, importance_dict):
    model = copy.deepcopy(model).to(DEVICE)
    model.eval()

    example_inputs = torch.randn(1, 3, 32, 32).to(DEVICE)

    DG = tp.DependencyGraph().build_dependency(
        model,
        example_inputs=example_inputs
    )

    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Conv2d) and name in importance_dict:
            imp = importance_dict[name].to(DEVICE)

            keep = int(module.out_channels * (1 - PRUNE_RATIO))
            prune = module.out_channels - keep

            if prune <= 0:
                continue

            # Converted to python list for torch_pruning compatibility
            idx = torch.argsort(imp)[:prune].tolist()

            group = DG.get_pruning_group(
                module,
                tp.prune_conv_out_channels,
                idxs=idx
            )

            if DG.check_pruning_group(group):
                group.prune()

    return model

# THIS WAS MISSING: Executing the pruning process using Taylor Importance
print("Executing Structural Pruning...")
pruned_model = prune_structural(model, taylor_final)

params_after = count_params(pruned_model)

print("Parameters after pruning:", params_after)
print("Compression:", (1 - params_after/params_before)*100,"%")
PRUNED_INITIAL_PATH = "/kaggle/working/resnet50_pruned_initial100epochs.pt"

torch.save({
    "model_state_dict": pruned_model.state_dict(),
    "params": count_params(pruned_model),
    "prune_ratio": PRUNE_RATIO
}, PRUNED_INITIAL_PATH)

print("✅ Structurally pruned model saved (before FT)")

Executing Structural Pruning...
Parameters after pruning: 7207377
Compression: 69.59586424139258 %
✅ Structurally pruned model saved (before FT)


In [8]:
optimizer = torch.optim.SGD(
    pruned_model.parameters(),
    lr=0.05, momentum=0.9,
    weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS_FINETUNE)

BEST_PRUNED_PATH = "/kaggle/working/resnet50_pruned_finetuned_cifar100_best100epochs.pt"

best_acc = 0
print("Starting Fine-tuning...")
for epoch in range(EPOCHS_FINETUNE):

    pruned_model.train()
    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(pruned_model(x),y)
        loss.backward()
        optimizer.step()

    scheduler.step()
    acc = evaluate(pruned_model)

    print(f"FT Epoch {epoch+1} | Acc {acc*100:.2f}%")

    if acc > best_acc:
        best_acc = acc

        torch.save({
            "model_state_dict": pruned_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "accuracy": best_acc,
            "params": count_params(pruned_model)
        }, BEST_PRUNED_PATH)

        print("✅ Saved new best fine-tuned model")

print("\n===== FINAL RESULTS =====")
print("Baseline Params :", params_before)
print("Pruned Params   :", params_after)
print("Compression     :", (1 - params_after/params_before)*100,"%")
print("Final Accuracy  :", best_acc*100,"%")

Starting Fine-tuning...
FT Epoch 1 | Acc 41.80%
✅ Saved new best fine-tuned model
FT Epoch 2 | Acc 48.12%
✅ Saved new best fine-tuned model
FT Epoch 3 | Acc 49.56%
✅ Saved new best fine-tuned model
FT Epoch 4 | Acc 51.31%
✅ Saved new best fine-tuned model
FT Epoch 5 | Acc 53.57%
✅ Saved new best fine-tuned model
FT Epoch 6 | Acc 54.57%
✅ Saved new best fine-tuned model
FT Epoch 7 | Acc 51.65%
FT Epoch 8 | Acc 56.50%
✅ Saved new best fine-tuned model
FT Epoch 9 | Acc 55.85%
FT Epoch 10 | Acc 55.34%
FT Epoch 11 | Acc 58.47%
✅ Saved new best fine-tuned model
FT Epoch 12 | Acc 55.90%
FT Epoch 13 | Acc 56.58%
FT Epoch 14 | Acc 55.74%
FT Epoch 15 | Acc 49.69%
FT Epoch 16 | Acc 57.32%
FT Epoch 17 | Acc 51.34%
FT Epoch 18 | Acc 60.25%
✅ Saved new best fine-tuned model
FT Epoch 19 | Acc 54.15%
FT Epoch 20 | Acc 55.22%
FT Epoch 21 | Acc 57.87%
FT Epoch 22 | Acc 60.16%
FT Epoch 23 | Acc 59.54%
FT Epoch 24 | Acc 58.24%
FT Epoch 25 | Acc 60.21%
FT Epoch 26 | Acc 57.57%
FT Epoch 27 | Acc 59.33%
FT E

In [9]:
from sklearn.metrics import precision_recall_fscore_support
import numpy as np

def evaluate_comprehensive(model, dataloader):
    model.eval()
    
    top1_correct = 0
    top5_correct = 0
    total = 0
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            outputs = model(x)
            
            # Calculate Top-1 and Top-5 predictions
            _, pred = outputs.topk(5, 1, True, True)
            pred = pred.t() # Transpose for easier comparison
            
            # Check if predictions match the target
            correct = pred.eq(y.view(1, -1).expand_as(pred))
            
            # Top-1 is just the first row
            top1_correct += correct[0].sum().item()
            # Top-5 is any correct prediction in the top 5 rows
            top5_correct += correct[:5].sum().item()
            total += y.size(0)
            
            # Store Top-1 predictions and targets for sklearn metrics
            all_preds.extend(pred[0].cpu().numpy())
            all_targets.extend(y.cpu().numpy())
            
    top1_acc = top1_correct / total * 100
    top5_acc = top5_correct / total * 100
    
    # Calculate Macro Precision, Recall, and F1-Score
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average='macro', zero_division=0
    )
    
    print("\n===== COMPREHENSIVE TEST METRICS =====")
    print(f"Top-1 Accuracy : {top1_acc:.2f}%")
    print(f"Top-5 Accuracy : {top5_acc:.2f}%")
    print(f"Macro Precision: {precision * 100:.2f}%")
    print(f"Macro Recall   : {recall * 100:.2f}%")
    print(f"Macro F1-Score : {f1 * 100:.2f}%")
    
    return top1_acc, top5_acc, precision, recall, f1

# Run the evaluation on your final model
print("Evaluating final pruned model on Test Set...")
evaluate_comprehensive(pruned_model, test_loader)

Evaluating final pruned model on Test Set...

===== COMPREHENSIVE TEST METRICS =====
Top-1 Accuracy : 74.85%
Top-5 Accuracy : 92.94%
Macro Precision: 74.83%
Macro Recall   : 74.85%
Macro F1-Score : 74.72%


(74.85000000000001,
 92.94,
 0.7482823999493431,
 0.7484999999999999,
 0.7472390307029196)